In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)

Project root: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project


In [2]:
from rdflib import Graph

g = Graph()
g.parse("../kg_artifacts/full_kb.ttl", format="turtle")

print(f"Total triples: {len(g)}")

# affiche quelques triples pour voir le format
for i, (s, p, o) in enumerate(g):
    print(f"{str(s).split('/')[-1]:30} | {str(p).split('/')[-1]:25} | {str(o).split('/')[-1][:40]}")
    if i > 10:
        break

Total triples: 42219
Q487485                        | P569                      | 1911-11-11T00:00:00+00:00
Q6948197                       | P21                       | Q6581072
Q5940928                       | P641                      | Q31920
Q4720573                       | P27                       | Q159
Q6673104                       | P641                      | Q31920
Q3893747                       | P569                      | 1954-06-07T00:00:00+00:00
Kick_Sauber                    | alignmentConfidence       | 0.8
Q9030016                       | P569                      | 1985-03-04T00:00:00+00:00
Q548585                        | P641                      | Q7707
Q5516798                       | P1344                     | Q8577
Q261657                        | P27                       | Q29
Q7929622                       | P1344                     | Q8567


In [3]:
from rdflib import URIRef, Literal

# filtre : garde seulement les triples (uri, uri, uri) — pas de literals
triples = []

for s, p, o in g:
    if isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef):
        s_str = str(s).split("/")[-1]
        p_str = str(p).split("/")[-1]
        o_str = str(o).split("/")[-1]
        triples.append((s_str, p_str, o_str))

print(f"Triples URI uniquement: {len(triples)}")
print("\nExemples:")
for t in triples[:10]:
    print(f"  {t[0]:25} | {t[1]:15} | {t[2]}")

Triples URI uniquement: 37376

Exemples:
  Q6948197                  | P21             | Q6581072
  Q5940928                  | P641            | Q31920
  Q4720573                  | P27             | Q159
  Q6673104                  | P641            | Q31920
  Q548585                   | P641            | Q7707
  Q5516798                  | P1344           | Q8577
  Q261657                   | P27             | Q29
  Q7929622                  | P1344           | Q8567
  Q7522390                  | P1344           | Q8577
  Q4297386                  | P1344           | Q8577


In [4]:
import random

# mélange aléatoire reproductible
random.seed(42)
random.shuffle(triples)

total = len(triples)
train_end = int(total * 0.8)
valid_end = int(total * 0.9)

train = triples[:train_end]
valid = triples[train_end:valid_end]
test  = triples[valid_end:]

print(f"Train : {len(train)}")
print(f"Valid : {len(valid)}")
print(f"Test  : {len(test)}")

Train : 29900
Valid : 3738
Test  : 3738


In [5]:
from pathlib import Path

kge_dir = Path("../data/kge")
kge_dir.mkdir(exist_ok=True)

for split_name, split_data in [("train", train), ("valid", valid), ("test", test)]:
    path = kge_dir / f"{split_name}.txt"
    with open(path, "w", encoding="utf-8") as f:
        for s, p, o in split_data:
            f.write(f"{s}\t{p}\t{o}\n")
    print(f"Sauvegardé → {path} ({len(split_data)} triples)")

Sauvegardé → ..\data\kge\train.txt (29900 triples)
Sauvegardé → ..\data\kge\valid.txt (3738 triples)
Sauvegardé → ..\data\kge\test.txt (3738 triples)


In [6]:
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

# charge les triples
tf = TriplesFactory.from_path("../data/kge/train.txt")
tf_valid = TriplesFactory.from_path(
    "../data/kge/valid.txt",
    entity_to_id=tf.entity_to_id,
    relation_to_id=tf.relation_to_id,
)
tf_test = TriplesFactory.from_path(
    "../data/kge/test.txt",
    entity_to_id=tf.entity_to_id,
    relation_to_id=tf.relation_to_id,
)

print(f"Entités  : {tf.num_entities}")
print(f"Relations: {tf.num_relations}")
print(f"Train    : {tf.num_triples}")

You're trying to map triples with 470 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 463 from 3738 triples were filtered out
You're trying to map triples with 462 entities and 1 relations that are not in the training set. These triples will be excluded from the mapping.
In total 453 from 3738 triples were filtered out


Entités  : 10931
Relations: 20
Train    : 29900


In [7]:
from pykeen.pipeline import pipeline

print("Entraînement TransE...")

result_transe = pipeline(
    training=tf,
    validation=tf_valid,
    testing=tf_test,
    model="TransE",
    model_kwargs={"embedding_dim": 100},
    optimizer="Adam",
    optimizer_kwargs={"lr": 0.01},
    training_kwargs={
        "num_epochs": 50,
        "batch_size": 256,
    },
    evaluation_kwargs={"batch_size": 256},
    random_seed=42,
    device="cpu",
)

print("\nRésultats TransE:")
print(result_transe.metric_results.to_df())

Entraînement TransE...


c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training epochs on cpu:   0%|          | 0/50 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/117 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/117 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/117 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/117 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/3.29k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 26.63s seconds



Résultats TransE:
     Side    Rank_type              Metric         Value
0    head   optimistic            variance  4.654930e+06
1    tail   optimistic            variance  1.285251e+06
2    both   optimistic            variance  3.693845e+06
3    head    realistic            variance  4.654930e+06
4    tail    realistic            variance  1.285252e+06
..    ...          ...                 ...           ...
220  tail    realistic  adjusted_hits_at_k  2.733077e-01
221  both    realistic  adjusted_hits_at_k  1.457111e-01
222  head  pessimistic  adjusted_hits_at_k  1.808978e-02
223  tail  pessimistic  adjusted_hits_at_k  2.733077e-01
224  both  pessimistic  adjusted_hits_at_k  1.457111e-01

[225 rows x 4 columns]


In [8]:
# extrait les métriques clés
metrics_df = result_transe.metric_results.to_df()

# filtre les métriques importantes
key_metrics = metrics_df[
    (metrics_df["Side"] == "both") & 
    (metrics_df["Rank_type"] == "realistic") &
    (metrics_df["Metric"].isin(["hits_at_1", "hits_at_3", "hits_at_10", "inverse_harmonic_mean_rank"]))
]

print("TransE — Métriques clés:")
print(key_metrics[["Metric", "Value"]].to_string(index=False))

TransE — Métriques clés:
                    Metric    Value
inverse_harmonic_mean_rank 0.057241
                 hits_at_1 0.012329
                 hits_at_3 0.060426
                hits_at_10 0.146575


In [9]:
print("Entraînement DistMult...")

result_distmult = pipeline(
    training=tf,
    validation=tf_valid,
    testing=tf_test,
    model="DistMult",
    model_kwargs={"embedding_dim": 100},
    optimizer="Adam",
    optimizer_kwargs={"lr": 0.01},
    training_kwargs={
        "num_epochs": 50,
        "batch_size": 256,
    },
    evaluation_kwargs={"batch_size": 256},
    random_seed=42,
    device="cpu",
)

metrics_df2 = result_distmult.metric_results.to_df()
key_metrics2 = metrics_df2[
    (metrics_df2["Side"] == "both") & 
    (metrics_df2["Rank_type"] == "realistic") &
    (metrics_df2["Metric"].isin(["hits_at_1", "hits_at_3", "hits_at_10", "inverse_harmonic_mean_rank"]))
]

print("\nDistMult — Métriques clés:")
print(key_metrics2[["Metric", "Value"]].to_string(index=False))

INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding(
  (regularizer): LpRegularizer()
)


Entraînement DistMult...


c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training epochs on cpu:   0%|          | 0/50 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/117 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/117 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/117 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/3.29k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 14.75s seconds



DistMult — Métriques clés:
                    Metric    Value
inverse_harmonic_mean_rank 0.271662
                 hits_at_1 0.209285
                 hits_at_3 0.293760
                hits_at_10 0.384475


In [10]:
from pathlib import Path
import pandas as pd

kge_dir = Path("../data/kge")

# sauvegarde les modèles
result_transe.save_to_directory(str(kge_dir / "transe_model"))
result_distmult.save_to_directory(str(kge_dir / "distmult_model"))

# sauvegarde les métriques dans un CSV pour le rapport
results_summary = pd.DataFrame([
    {
        "model": "TransE",
        "MRR":      0.057241,
        "Hits@1":   0.012329,
        "Hits@3":   0.060426,
        "Hits@10":  0.146575,
    },
    {
        "model": "DistMult",
        "MRR":      0.271662,
        "Hits@1":   0.209285,
        "Hits@3":   0.293760,
        "Hits@10":  0.384475,
    }
])

results_summary.to_csv(kge_dir / "evaluation_results.csv", index=False)
print("Sauvegardé ✅")
print(results_summary.to_string(index=False))

INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=10931, num_relations=20, create_inverse_triples=False, num_triples=29900, path="C:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\data\kge\train.txt") to file:///C:/Users/paula/Documents/Cours/A4/S2/WebData/Project/f1_semantic_web_project/data/kge/transe_model/training_triples
INFO:pykeen.pipeline.api:Saved to directory: C:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\data\kge\transe_model
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=10931, num_relations=20, create_inverse_triples=False, num_triples=29900, path="C:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\data\kge\train.txt") to file:///C:/Users/paula/Documents/Cours/A4/S2/WebData/Project/f1_semantic_web_project/data/kge/distmult_model/training_triples
INFO:pykeen.pipeline.api:Saved to directory: C:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_s

Sauvegardé ✅
   model      MRR   Hits@1   Hits@3  Hits@10
  TransE 0.057241 0.012329 0.060426 0.146575
DistMult 0.271662 0.209285 0.293760 0.384475


In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from rdflib import Graph, URIRef
from rdflib.namespace import RDF

# récupère les embeddings DistMult
entity_embeddings = result_distmult.model.entity_representations[0]
embeddings_matrix = entity_embeddings().detach().numpy()

print(f"Shape embeddings: {embeddings_matrix.shape}")